In [26]:
import numpy as np
import pandas as pd
import pickle

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
df = pd.read_csv("Dataset/AAPL_2006-01-01_to_2018-01-01.csv")
df.head()

,Date,Open,High,Low,Close,Volume,Name
0,2006-01-03,10.34,10.68,10.32,10.68,201853036,AAPL
1,2006-01-04,10.73,10.85,10.64,10.71,155225609,AAPL
2,2006-01-05,10.69,10.70,10.54,10.63,112396081,AAPL
3,2006-01-06,10.75,10.96,10.65,10.90,176139334,AAPL
4,2006-01-09,10.96,11.03,10.82,10.86,168861224,AAPL


In [ ]:
print("Dataset Shape:", df.shape)

print("Columns:")
print(df.columns.tolist())

print("Missing Values:")
print(df.isnull().sum())

Dataset Shape: (3019, 7)

Columns:
['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Name']

Missing Values:
Date      0
Open      0
High      0
Low       0
Close     0
Volume    0
Name      0
dtype: int64


In [ ]:
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date")
df = df.dropna()
df = df.reset_index(drop=True)

print(df.head())
print("Dataset Shape:", df.shape)

        Date   Open   High    Low  Close     Volume  Name
0 2006-01-03  10.34  10.68  10.32  10.68  201853036  AAPL
1 2006-01-04  10.73  10.85  10.64  10.71  155225609  AAPL
2 2006-01-05  10.69  10.70  10.54  10.63  112396081  AAPL
3 2006-01-06  10.75  10.96  10.65  10.90  176139334  AAPL
4 2006-01-09  10.96  11.03  10.82  10.86  168861224  AAPL

Dataset Shape: (3019, 7)


In [ ]:
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date")
df = df.dropna()
df = df.reset_index(drop=True)

print(df.head())
print("Dataset Shape:", df.shape)

        Date   Open   High    Low  Close     Volume  Name
0 2006-01-03  10.34  10.68  10.32  10.68  201853036  AAPL
1 2006-01-04  10.73  10.85  10.64  10.71  155225609  AAPL
2 2006-01-05  10.69  10.70  10.54  10.63  112396081  AAPL
3 2006-01-06  10.75  10.96  10.65  10.90  176139334  AAPL
4 2006-01-09  10.96  11.03  10.82  10.86  168861224  AAPL

Dataset Shape: (3019, 7)


In [ ]:
data = df[["Close"]].copy()
print(data.head())

   Close
0  10.68
1  10.71
2  10.63
3  10.90
4  10.86


In [ ]:
train_size = int(len(data) * 0.8)
train_data = data.iloc[:train_size]
test_data = data.iloc[train_size:]

print("Total data:", len(data))
print("Training data:", len(train_data))
print("Testing data:", len(test_data))

Total data: 3019
Training data: 2415
Testing data: 604


In [ ]:
scaler = MinMaxScaler(feature_range=(0, 1))
train_scaled = scaler.fit_transform(train_data)
test_scaled = scaler.transform(test_data)

In [ ]:
SEQUENCE_LENGTH = 60
def create_sequences(data, sequence_length):
    X = []
    y = []
    for i in range(sequence_length, len(data)):
        X.append(data[i - sequence_length : i])
        y.append(data[i])
    return np.array(X), np.array(y)

In [ ]:
X_train, y_train = create_sequences(train_scaled, SEQUENCE_LENGTH)
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

X_train shape: (2355, 60, 1)
y_train shape: (2355, 1)


In [ ]:
combined_data = np.concatenate((train_scaled[-SEQUENCE_LENGTH:], test_scaled), axis=0)

In [ ]:
X_test, y_test = create_sequences(combined_data, SEQUENCE_LENGTH)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_test shape: (604, 60, 1)
y_test shape: (604, 1)


In [ ]:
model = Sequential(
    [
        LSTM(64, return_sequences=True, input_shape=(SEQUENCE_LENGTH, 1)),
        LSTM(32),
        Dense(1),
    ]
)

/home/anas/LEARNING/GEN-AI/python/venv/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
model.compile(optimizer="adam", loss="mean_squared_error")

In [42]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 60, 64)         │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 29,345 (114.63 KB)

 Trainable params: 29,345 (114.63 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stopping],
    verbose=1,
)

Epoch 1/30
67/67 ━━━━━━━━━━━━━━━━━━━━ 11s 79ms/step - loss: 0.0071 - val_loss: 0.0013
Epoch 2/30
67/67 ━━━━━━━━━━━━━━━━━━━━ 5s 69ms/step - loss: 3.0409e-04 - val_loss: 0.0015
Epoch 3/30
67/67 ━━━━━━━━━━━━━━━━━━━━ 5s 72ms/step - loss: 3.0626e-04 - val_loss: 0.0015
Epoch 4/30
67/67 ━━━━━━━━━━━━━━━━━━━━ 5s 78ms/step - loss: 3.1393e-04 - val_loss: 0.0016
Epoch 5/30
67/67 ━━━━━━━━━━━━━━━━━━━━ 5s 73ms/step - loss: 2.7564e-04 - val_loss: 7.9368e-04
Epoch 6/30
67/67 ━━━━━━━━━━━━━━━━━━━━ 7s 107ms/step - loss: 2.6477e-04 - val_loss: 8.1677e-04
Epoch 7/30
67/67 ━━━━━━━━━━━━━━━━━━━━ 5s 72ms/step - loss: 2.5206e-04 - val_loss: 0.0011
Epoch 8/30
67/67 ━━━━━━━━━━━━━━━━━━━━ 5s 71ms/step - loss: 2.9136e-04 - val_loss: 0.0020
Epoch 9/30
67/67 ━━━━━━━━━━━━━━━━━━━━ 6s 92ms/step - loss: 2.5555e-04 - val_loss: 8.3457e-04
Epoch 10/30
67/67 ━━━━━━━━━━━━━━━━━━━━ 9s 76ms/step - loss: 2.2359e-04 - val_loss: 6.4641e-04
Epoch 11/30
67/67 ━━━━━━━━━━━━━━━━━━━━ 6s 86ms/step - loss: 2.3894e-04 - val_loss: 7.1292e-04
E

In [ ]:
predictions_scaled = model.predict(X_test)
print("Prediction shape:", predictions_scaled.shape)

19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step
Prediction shape: (604, 1)


In [ ]:
predictions = scaler.inverse_transform(predictions_scaled)
actual_prices = scaler.inverse_transform(y_test)

In [ ]:
mae = mean_absolute_error(actual_prices, predictions)
mse = mean_squared_error(actual_prices, predictions)
rmse = np.sqrt(mse)

In [50]:
print(f"MAE  : {mae:.4f}")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")

MAE  : 2.9494
MSE  : 15.1717
RMSE : 3.8951


In [ ]:
results = pd.DataFrame(
    {"Actual Price": actual_prices.flatten(), "Predicted Price": predictions.flatten()}
)
results.head(20)

,Actual Price,Predicted Price
0,119.72,119.248901
1,113.49,118.683388
2,115.24,117.922310
3,115.15,117.314461
4,115.96,116.817680
5,117.16,116.496620
6,116.50,116.401421
7,115.01,116.391670
8,112.65,116.326202
9,105.76,116.056541


In [53]:
model.save("model.keras")

In [54]:
with open("scaler.pkl", "wb") as file:
    pickle.dump(scaler, file)